# The Essential Python Libraries for Data Science

**A guided tour of the 9 libraries that do 95% of the work in a modern data team.**

---

### How this notebook is organized

We'll follow the natural arc of a data science project, and meet each library *where it actually shows up in the workflow*:

| Stage | Library | What it does |
|---|---|---|
| **1. Data wrangling** | `pandas` | Load, clean, reshape tabular data |
| **2. Visualization** | `matplotlib`, `seaborn`, `plotly` | Static and interactive charts |
| **3. Classical ML** | `scikit-learn` | Predictive modeling, the workhorse |
| **4. Statistics** | `statsmodels` | Inference, p-values, confidence intervals |
| **5. Deep learning** | `pytorch` | Neural networks for unstructured data |
| **6. NLP / pretrained models** | `huggingface` | State-of-the-art text & vision models |
| **7. Serving models** | `fastapi` | Wrap a model as a web API |
| **8. Internal apps** | `streamlit` | Build a dashboard in 20 lines |

We'll use a consistent insurance dataset throughout so you can see how the same data flows through each layer of the stack.

> **How to run:** Click each cell and press **Shift + Enter**. Run cells in order — later cells depend on earlier ones.

## 0. Setup

Most of these libraries come pre-installed in Anaconda, Google Colab, and Deepnote. If you're running locally and a cell fails with `ModuleNotFoundError`, uncomment the relevant line below.

In [ ]:
# !pip install pandas matplotlib seaborn plotly scikit-learn statsmodels
# !pip install torch transformers fastapi uvicorn streamlit

# Sanity check — confirm what's available
import importlib
libs = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly',
        'sklearn', 'statsmodels', 'torch', 'transformers',
        'fastapi', 'streamlit']

for lib in libs:
    try:
        m = importlib.import_module(lib)
        version = getattr(m, '__version__', 'installed')
        print(f"  ✅  {lib:15s} {version}")
    except ImportError:
        print(f"  ⚠️   {lib:15s} not installed (we'll show its code anyway)")

## 1. `pandas` — The data wrangler

> *"80% of data science is cleaning data. Pandas is what you use to do it."*

**What it is:** A library for working with **tabular data** (think: spreadsheets, but programmable). The core object is the `DataFrame` — a table with named columns and an index.

**Why it matters:** Almost every data science project starts and ends with pandas. It's the lingua franca that connects databases, CSVs, Excel files, APIs, and ML libraries.

In [ ]:
import pandas as pd
import numpy as np

# Generate a realistic insurance dataset we'll reuse throughout this notebook
np.random.seed(42)
n = 1500

df = pd.DataFrame({
    'policy_id': range(10001, 10001 + n),
    'age': np.random.randint(18, 75, n),
    'bmi': np.clip(np.random.normal(28, 6, n), 16, 50).round(1),
    'children': np.random.choice([0, 1, 2, 3, 4], n, p=[0.42, 0.24, 0.20, 0.10, 0.04]),
    'smoker': np.random.choice(['yes', 'no'], n, p=[0.20, 0.80]),
    'region': np.random.choice(['NE', 'SE', 'SW', 'NW'], n),
    'claims_filed': np.random.poisson(0.7, n),
})

# Charges depend on age, bmi, smoking — realistic actuarial pattern
df['annual_charges'] = (
    250 * df['age']
    + 350 * (df['bmi'] - 25).clip(lower=0)
    + 22000 * (df['smoker'] == 'yes').astype(int)
    + 500 * df['children']
    + np.random.normal(0, 2500, n)
).clip(lower=1000).round(2)

print(f"DataFrame shape: {df.shape}")
df.head()

### The five pandas operations you'll use every day

These five patterns cover the majority of real-world pandas work.

In [ ]:
# 1. FILTERING — pull rows that match a condition
high_risk = df[(df['smoker'] == 'yes') & (df['bmi'] > 30)]
print(f"1. Filter: {len(high_risk)} high-risk customers (smokers with BMI > 30)")

# 2. GROUPBY — split-apply-combine, the heart of analysis
by_region = df.groupby('region').agg(
    customers=('policy_id', 'count'),
    avg_charges=('annual_charges', 'mean'),
    total_claims=('claims_filed', 'sum')
).round(0)
print("\n2. Group by region:")
print(by_region)

In [ ]:
# 3. CREATING NEW COLUMNS — feature engineering
df['risk_tier'] = pd.cut(
    df['annual_charges'],
    bins=[0, 10000, 25000, 100000],
    labels=['Preferred', 'Standard', 'High-Risk']
)
print("3. New column — risk tier distribution:")
print(df['risk_tier'].value_counts())

# 4. PIVOT TABLES — Excel pivot tables, but programmable
pivot = df.pivot_table(
    values='annual_charges',
    index='region',
    columns='smoker',
    aggfunc='mean'
).round(0)
print("\n4. Pivot — avg charges by region × smoker:")
print(pivot)

In [ ]:
# 5. JOINING / MERGING — combining multiple tables
# Imagine a separate table from the marketing system
marketing = pd.DataFrame({
    'policy_id': df['policy_id'].sample(800, random_state=1).values,
    'acquisition_channel': np.random.choice(['web', 'broker', 'referral'], 800),
})

enriched = df.merge(marketing, on='policy_id', how='left')
print("5. Merge result — rows with channel info:")
print(enriched['acquisition_channel'].value_counts(dropna=False))

> 💡 **Mental model:** If you can do it in Excel — sort, filter, pivot, vlookup, formulas — you can do it 1000× faster on 100× more data with pandas. Everything else in this notebook builds on top of pandas.

## 2. `matplotlib` + `seaborn` — Static visualization

**Matplotlib** is the foundational plotting library — flexible but verbose. **Seaborn** is built on top of matplotlib and provides a higher-level, statistics-aware API with prettier defaults.

In practice, most teams use **seaborn for quick exploration** and drop down to **matplotlib for fine-grained customization**.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Histogram — distribution of a single variable
sns.histplot(data=df, x='annual_charges', hue='smoker', bins=40,
             ax=axes[0], palette=['#06A77D', '#D62246'])
axes[0].set_title('Distribution: charges by smoker status')

# Scatter — relationship between two variables
sns.scatterplot(data=df, x='age', y='annual_charges', hue='smoker',
                ax=axes[1], alpha=0.5, palette=['#06A77D', '#D62246'])
axes[1].set_title('Scatter: age vs. charges')

# Box plot — distribution across categories
sns.boxplot(data=df, x='region', y='annual_charges',
            ax=axes[2], hue='region', legend=False, palette='Set2')
axes[2].set_title('Box plot: charges by region')

plt.tight_layout()
plt.show()

In [ ]:
# Seaborn's killer feature: statistical plots in one line
g = sns.lmplot(
    data=df, x='age', y='annual_charges',
    hue='smoker', col='region', col_wrap=2,
    height=3, aspect=1.4,
    scatter_kws={'alpha': 0.4, 's': 15},
    palette=['#06A77D', '#D62246']
)
g.fig.suptitle('Age vs. charges, with regression lines, faceted by region', y=1.02)
plt.show()

> 💡 **When to use which:**
> - **Seaborn** — exploring data, quick statistical plots, presentations
> - **Matplotlib** — when you need pixel-perfect control (publication figures, custom annotations)
> - **Both together** — start in seaborn, refine the result with matplotlib commands

## 3. `plotly` — Interactive visualization

Where matplotlib/seaborn produce **static images**, Plotly produces **interactive HTML charts** you can zoom, pan, hover, and export. It's the go-to library for dashboards and shareable analyses.

In [ ]:
import plotly.express as px

# Same data, but now interactive — try hovering, zooming, clicking the legend
fig = px.scatter(
    df, x='age', y='annual_charges',
    color='smoker', size='bmi',
    hover_data=['policy_id', 'region', 'children'],
    title='Insurance customers — age vs. charges (hover over points!)',
    color_discrete_map={'yes': '#D62246', 'no': '#06A77D'},
    opacity=0.6,
    height=500
)
fig.show()

In [ ]:
# Plotly excels at multi-dimensional views
# Sunburst chart — hierarchical breakdown of the book of business
agg = df.groupby(['region', 'smoker', 'risk_tier'], observed=True).size().reset_index(name='count')

fig = px.sunburst(
    agg, path=['region', 'smoker', 'risk_tier'], values='count',
    title='Book of business breakdown (click slices to zoom in)',
    height=500
)
fig.show()

> 💡 **When to use Plotly:**
> - **Dashboards & web apps** — Plotly powers Streamlit, Dash, and many internal tools
> - **Sharing analyses** — embed in HTML emails, Confluence, Notion
> - **Multi-dimensional data** — 3D plots, sunbursts, sankey diagrams
>
> For static reports and academic papers, stick with matplotlib/seaborn.

## 4. `scikit-learn` — Classical machine learning

The single most important ML library in Python. It implements virtually every classical algorithm — regression, classification, clustering, dimensionality reduction — behind a **consistent API**.

The pattern is always the same:
1. `model = SomeModel(...)`
2. `model.fit(X, y)`
3. `predictions = model.predict(X_new)`

Once you've learned the pattern for one algorithm, you've learned it for all of them.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# Business problem: predict which customers will file a claim this year
df['will_claim'] = (df['claims_filed'] > 0).astype(int)

X = pd.get_dummies(df[['age', 'bmi', 'children', 'smoker', 'region']], drop_first=True)
y = df['will_claim']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Train: {len(X_train)} customers  |  Test: {len(X_test)} customers")
print(f"Base rate (anyone files a claim?): {y.mean():.1%}")

In [ ]:
# Pipelines — sklearn's killer feature.
# Bundle preprocessing + model so they travel together (no leakage, easy to deploy)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingClassifier(random_state=42))
])

pipe.fit(X_train, y_train)

# Cross-validation: train/test on 5 different splits, report stability
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
print(f"5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Held-out test performance
test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"Held-out test AUC: {test_auc:.3f}\n")
print(classification_report(y_test, pipe.predict(X_test)))

> 💡 **Why scikit-learn dominates:**
> - **Consistent API** across hundreds of algorithms
> - **Pipelines** prevent the most common ML bug: data leakage
> - **Battle-tested** — it's been in production at thousands of companies for over a decade
>
> If your problem fits in a pandas DataFrame, scikit-learn is almost always the right starting point — even before reaching for deep learning.

## 5. `statsmodels` — Statistical inference

**Scikit-learn answers "what will happen?"** — it's optimized for prediction.
**Statsmodels answers "why does it happen, and how confident are we?"** — it's optimized for inference.

If you need **p-values, confidence intervals, and standard errors** (e.g. for a regulatory filing, an A/B test, or a peer-reviewed paper), this is the library.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# R-style formula syntax — incredibly readable
model = smf.ols('annual_charges ~ age + bmi + children + C(smoker) + C(region)', data=df).fit()
print(model.summary().tables[1])

**How to read this output (the actuarial classic):**

| Column | Meaning |
|---|---|
| `coef` | Effect on charges per unit of the variable, holding others constant |
| `std err` | Uncertainty in that estimate |
| `P>\|t\|` | p-value — probability the effect is actually zero |
| `[0.025, 0.975]` | 95% confidence interval |

For example, `C(smoker)[T.yes]` having a coefficient around `+22,000` with `p < 0.001` means: *"Smokers cost roughly $22,000 more per year, and we are highly confident this effect is real."*

> 💡 **When you need statsmodels over scikit-learn:**
> - **Regulated pricing** — regulators want p-values and confidence intervals
> - **A/B test analysis** — proper statistical tests with CIs
> - **Time series** — ARIMA, state space models, etc. are best-in-class
> - **Causal inference** — instrumental variables, fixed effects, etc.

## 6. `pytorch` — Deep learning

The dominant framework for **neural networks**, used by OpenAI, Meta, Tesla, and most academic ML research.

**When you actually need it:**
- **Unstructured data** — images, audio, text, video
- **Very large datasets** — millions+ of examples
- **Custom architectures** — when off-the-shelf models don't fit

**When you don't:** for tabular data with < 100k rows, scikit-learn (especially gradient boosting) usually beats deep learning while being simpler and faster.

For demonstration, we'll build a tiny neural net that predicts claim probability from our insurance features — even though scikit-learn would do it better.

In [ ]:
import torch
import torch.nn as nn

# Convert numpy → tensors (PyTorch's core data structure)
X_train_t = torch.FloatTensor(StandardScaler().fit_transform(X_train).astype(float))
y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1)

# Define a simple neural network
class ClaimPredictor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = ClaimPredictor(n_features=X_train_t.shape[1])

# The training loop — this pattern is at the heart of all deep learning
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

losses = []
for epoch in range(100):
    optimizer.zero_grad()           # reset gradients
    preds = model(X_train_t)        # forward pass
    loss = loss_fn(preds, y_train_t)# compute loss
    loss.backward()                 # backprop — compute gradients
    optimizer.step()                # update weights
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
print(f"Improvement:  {(1 - losses[-1]/losses[0])*100:.0f}%")

# Visualize training
plt.figure(figsize=(9, 3.5))
plt.plot(losses, color='#2E86AB', linewidth=2)
plt.title('Neural network training — loss over epochs')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.tight_layout()
plt.show()

> 💡 **The four PyTorch concepts you must understand:**
> 1. **Tensors** — n-dimensional arrays that can run on GPUs
> 2. **`nn.Module`** — the base class for any neural network
> 3. **Autograd** — automatic differentiation (PyTorch tracks gradients for you)
> 4. **The training loop** — forward → loss → backward → step (the 4 lines above)
>
> Everything else in deep learning is a variation on this theme.

## 7. Hugging Face `transformers` — Pretrained models

Training a deep learning model from scratch is expensive. **The 2020s breakthrough was that you almost never need to** — you can download a model someone else already trained on billions of examples, and either use it directly or fine-tune it on your problem.

**Hugging Face** is the GitHub of pretrained models. It hosts **800,000+ models** for text, vision, audio, and multimodal tasks, all behind a unified API.

The simplest entry point is the **`pipeline`** — load a state-of-the-art model in one line.

In [ ]:
# Note: this cell will download a model the first time you run it (~250 MB).
# If you don't have transformers installed, the code below shows what it would do.

try:
    from transformers import pipeline

    # Sentiment analysis on customer feedback — useful for claims processing,
    # complaint triage, and analyzing call-center transcripts
    sentiment = pipeline('sentiment-analysis')

    customer_feedback = [
        "The claims process was incredibly smooth and the agent was helpful.",
        "I've been waiting three weeks for a response. This is unacceptable.",
        "Premium increased without explanation. Considering switching providers.",
        "Quick payout after the accident — very impressed with the service.",
    ]

    results = sentiment(customer_feedback)
    for text, result in zip(customer_feedback, results):
        emoji = '😊' if result['label'] == 'POSITIVE' else '😞'
        print(f"{emoji} [{result['label']:8s} {result['score']:.2f}]  {text}")

except ImportError:
    print("transformers not installed — install with: pip install transformers")
    print("Then this cell would classify customer feedback as POSITIVE / NEGATIVE")
    print("using a state-of-the-art language model — in two lines of code.")

### Other tasks the same `pipeline` API covers

```python
pipeline('summarization')           # Summarize long claim narratives
pipeline('zero-shot-classification')# Categorize without training data
pipeline('question-answering')      # Extract answers from policy documents
pipeline('translation_en_to_fr')    # Translate customer communications
pipeline('image-classification')    # Classify damage photos for claims
pipeline('automatic-speech-recognition')  # Transcribe call recordings
```

> 💡 **Why this changed the industry:** Five years ago, building a sentiment classifier required a team of ML engineers, weeks of work, and labeled data. Today, the cell above does it in 3 lines, with better accuracy than that team could have achieved. **Pretrained models are the single biggest leverage point in modern AI.**

## 8. `fastapi` — Serving models as APIs

A trained model on your laptop creates zero business value. To matter, it has to be **callable from your other systems** — the website, the underwriting tool, the claims platform.

**FastAPI** wraps Python code as a production-grade web API in a few lines. It's what most teams reach for to put a model behind an HTTP endpoint.

In [ ]:
# Below is a working FastAPI app — you'd save this as `serve.py` and run:
#   uvicorn serve:app --reload
# Then any system on your network can POST customer features and get a prediction.

api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

# Load the trained model (saved earlier with joblib.dump)
model = joblib.load("claim_model.pkl")
app = FastAPI(title="Insurance Claim Predictor")

# Define the request schema — FastAPI validates incoming data automatically
class CustomerProfile(BaseModel):
    age: int
    bmi: float
    children: int
    smoker: str   # "yes" or "no"
    region: str

@app.post("/predict")
def predict_claim(profile: CustomerProfile):
    features = [[profile.age, profile.bmi, profile.children,
                 1 if profile.smoker == "yes" else 0]]
    probability = model.predict_proba(features)[0, 1]

    return {
        "claim_probability": round(float(probability), 3),
        "risk_tier": "high" if probability > 0.5 else "low",
        "model_version": "v1.0",
    }
'''
print(api_code)

**What this gives you for free:**
- Automatic input validation (the `CustomerProfile` schema)
- Auto-generated interactive docs at `http://localhost:8000/docs`
- JSON serialization
- Async support for high throughput
- Production-ready performance (it's used by Microsoft, Netflix, Uber)

> 💡 **The model lifecycle in production:**
> 1. **Train** the model in a notebook (sklearn, pytorch)
> 2. **Save** it to disk (`joblib.dump(model, ...)` or `torch.save(...)`)
> 3. **Wrap** it with FastAPI
> 4. **Deploy** to a server (Docker → Kubernetes / Cloud Run / SageMaker / etc.)
> 5. **Monitor** — track input distributions, prediction quality, latency
>
> Steps 4 and 5 — sometimes called **MLOps** — are where most projects struggle.

## 9. `streamlit` — Internal apps without a frontend team

**FastAPI** exposes a model to other *systems*. **Streamlit** exposes a model to other *humans* — without writing a line of HTML, CSS, or JavaScript.

You write a Python script. Streamlit turns it into a clickable web app. It's how a single data scientist can ship a dashboard a CFO will actually use, before lunch.

In [ ]:
# Save this as `app.py` and run: streamlit run app.py

streamlit_code = '''
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

st.set_page_config(page_title="Insurance Risk Explorer", layout="wide")
st.title("🏥 Insurance Risk Explorer")
st.markdown("Internal tool for underwriters — quote a customer in seconds.")

# Load the trained model and historical data
model = joblib.load("claim_model.pkl")
df = pd.read_parquet("policies.parquet")

# Sidebar — interactive inputs
st.sidebar.header("Customer profile")
age      = st.sidebar.slider("Age", 18, 75, 40)
bmi      = st.sidebar.slider("BMI", 16.0, 50.0, 27.0, step=0.5)
children = st.sidebar.slider("Children", 0, 5, 1)
smoker   = st.sidebar.selectbox("Smoker", ["no", "yes"])

# Main panel — live prediction
features = [[age, bmi, children, 1 if smoker == "yes" else 0]]
prob = model.predict_proba(features)[0, 1]

col1, col2, col3 = st.columns(3)
col1.metric("Claim probability", f"{prob:.1%}")
col2.metric("Risk tier", "High" if prob > 0.5 else "Low")
col3.metric("Suggested premium", f"${5000 + prob*30000:,.0f}")

# Embed an interactive chart
st.subheader("How this customer compares to the book")
fig = px.scatter(df, x="age", y="annual_charges", color="smoker", opacity=0.4)
fig.add_scatter(x=[age], y=[prob*30000], mode="markers",
                marker=dict(size=20, color="red", symbol="star"),
                name="This customer")
st.plotly_chart(fig, use_container_width=True)
'''

print(streamlit_code)
print("\n" + "="*60)
print("That's the entire app. Run with: streamlit run app.py")
print("Streamlit hosts a web server, opens your browser, and you're done.")

> 💡 **FastAPI vs. Streamlit — when to use which:**
>
> | | FastAPI | Streamlit |
> |---|---|---|
> | **Audience** | Other systems | Other humans |
> | **Output** | JSON | A web UI |
> | **Use case** | Production model serving | Internal dashboards, demos, prototypes |
> | **Scale** | Millions of requests/day | Tens of users |
> | **Build time** | Hours to days | Minutes |
>
> Most teams use **both** — FastAPI for the production API that powers the website, Streamlit for the internal tool the underwriters use to debug edge cases.

## The Full Stack at a Glance

Here's how the libraries fit together in a typical project:

```
┌─────────────────────────────────────────────────────────────┐
│  DATA SOURCES (databases, CSVs, APIs, S3, ...)              │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  pandas         ← load, clean, reshape, feature engineer    │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
              ┌────────────┴────────────┐
              ▼                         ▼
┌─────────────────────────┐  ┌─────────────────────────────┐
│  matplotlib / seaborn   │  │  plotly                     │
│  → static reports       │  │  → interactive dashboards   │
└─────────────────────────┘  └─────────────────────────────┘
                           ▼
              ┌────────────┴────────────┐
              ▼                         ▼
┌─────────────────────────┐  ┌─────────────────────────────┐
│  scikit-learn           │  │  statsmodels                │
│  → prediction           │  │  → inference, p-values      │
└─────────────────────────┘  └─────────────────────────────┘
                           ▼
              ┌────────────┴────────────┐
              ▼                         ▼
┌─────────────────────────┐  ┌─────────────────────────────┐
│  pytorch                │  │  hugging face transformers  │
│  → custom neural nets   │  │  → pretrained models        │
└─────────────────────────┘  └─────────────────────────────┘
                           ▼
              ┌────────────┴────────────┐
              ▼                         ▼
┌─────────────────────────┐  ┌─────────────────────────────┐
│  FastAPI                │  │  Streamlit                  │
│  → API for systems      │  │  → app for humans           │
└─────────────────────────┘  └─────────────────────────────┘
```

### A practical learning order

If you're just starting:

1. **`pandas`** — invest serious time here, it pays off forever
2. **`matplotlib` + `seaborn`** — enough to explore data confidently
3. **`scikit-learn`** — your default for any predictive problem
4. **`statsmodels`** — when you need rigorous statistics
5. **`plotly` + `streamlit`** — when you need to share results
6. **`fastapi`** — when something is going to production
7. **`pytorch` + `huggingface`** — when tabular methods aren't enough

> Most data scientists are paid mostly for what they do with the **first three**. The rest are leverage on top of that foundation.